In [64]:

import numpy as np
import matplotlib.pyplot as plt
# библиотека на основе которой были сделаны графики
# https://docs.bokeh.org/en/latest/docs/user_guide/basic/layouts.html
from bokeh.layouts import column
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
# Активируем вывод графиков в ноутбук
output_notebook()

# ----------------------------
# ПАРАМЕТРЫ СИГНАЛОВ
# ----------------------------

# Общее количество отсчетов для всех сигналов
N = 16  # Количество дискретных точек (отсчетов) для построения сигналов

# Амплитуды сигналов:
W1_A = 1.0  # Амплитуда дополнительной волны 1
W2_A = 1.0  # Амплитуда дополнительной волны 2
W3_A = 0.0  # Амплитуда дополнительной волны 3
W4_A = 0.0  # Амплитуда дополнительной волны 4

# Фазы сигналов в градусах:
W1_P_DEG = 90   # Фаза дополнительной волны 1
W2_P_DEG = 0   # Фаза дополнительной волны 2
W3_P_DEG = 0   # Фаза дополнительной волны 3
W4_P_DEG = 0   # Фаза дополнительной волны 4

# Количество периодов сигнала (K):
W1_K = 1  # Дополнительная волна 1
W2_K = 0  # Дополнительная волна 2
W3_K = 0  # Дополнительная волна 3
W4_K = 0 # Дополнительная волна 4

# ----------------------------
# ПОДГОТОВКА ДАННЫХ
# ----------------------------

# Создаем массив отсчетов (от 0 до N-1)
N_i = np.arange(N)  # [0, 1, 2, ..., N-1]
N_i_half = np.arange(N // 2 + 1)

# Конвертация фаз из градусов в радианы (требуется для тригонометрических функций)
W1_P_RAD = W1_P_DEG * np.pi / 180 # Формула преобразования: радианы = градусы * π/180
W2_P_RAD = W2_P_DEG * np.pi / 180
W3_P_RAD = W3_P_DEG * np.pi / 180
W4_P_RAD = W4_P_DEG * np.pi / 180

# ----------------------------
# ГЕНЕРАЦИЯ СИГНАЛОВ
# ----------------------------

# Генерация синусоиды:
# Формула: A * sin(2π * K * n/N + φ)
#   K - количество периодов
#   n - текущий отсчет
#   N - общее количество отсчетов
#   φ - фаза в радианах

# Используем sin вместо cos, но с фазой, отличающейся на π/2
# (cos(x) = sin(x + π/2), но здесь фаза задается параметром)
# Генерация  волн:
wave_1 = W1_A * np.sin(2 * np.pi * W1_K * N_i / N + W1_P_RAD)
wave_2 = W2_A * np.sin(2 * np.pi * W2_K * N_i / N + W2_P_RAD)
wave_3 = W3_A * np.sin(2 * np.pi * W3_K * N_i / N + W3_P_RAD)
wave_4 = W4_A * np.sin(2 * np.pi * W4_K * N_i / N + W4_P_RAD)

# ----------------------------
# СУММИРОВАНИЕ СИГНАЛОВ
# ----------------------------

# Сумма сгенерированных сигналов, в зависимости от примера количество сигналов может быть больше или меньше.
sum_wave = wave_1 + wave_2 + wave_3 + wave_4


# Произвести анализ частотного базиса. С выводом данных о нем.
F_b = 2
# DFT (прямое преобразование Фурье)
REX = np.zeros(N // 2 + 1)
IMX = np.zeros(N // 2 + 1)
sin_k = np.zeros(N)
cos_k = np.zeros(N)

for K in range(N // 2 + 1):
    for I in range(N):
        angle = 2 * np.pi * K * I / N
        REX[K] += sum_wave[I] * np.cos(angle)
        # IMX[K] -= sum_wave[I] * np.sin(angle)
        IMX[K] += sum_wave[I] * np.sin(angle)

        if K == F_b:
            cos_k[I] = np.cos(angle)
            sin_k[I] = np.sin(angle)
            
            # sinus[I] =        np.sin(angle)
            # Форматирование значений для выравнивания
            k_str = f"K = {K}"
            i_str = f"I = {I}"
            imx_str = f"IMX[K] = {IMX[K]:.5f}"
            rex_str = f"REX[K] = {REX[K]:.5f}"
            sin_mg = f"sin_mg = {sum_wave[I] * np.sin(angle):.5f}"
            cos_mg = f"cos_mg = {sum_wave[I] * np.cos(angle):.5f}"
            xx_str = f"sum[I] = {sum_wave[I]:.5f}"
            sin_str = f"np.sin(angle) = {np.sin(angle):.5f}"
            cos_str = f"np.cos(angle) = {np.cos(angle):.5f}"
            angle_str = f"angle = {(180 * angle / 3.14) % 360:.2f}°"
            # angle_str = f"{angle:.2f}"
 
            # Вывод с фиксированной шириной колонок
            print(f"{k_str:<8}{i_str:<8}{imx_str:<20}{sin_mg:<20}{rex_str:<20}{cos_mg:<20}{xx_str:<20}{sin_str:<25}{cos_str:<25}{angle_str:<5}")
print("Амплитудный и фазовый спектры = " + str(np.sqrt(REX**2 + IMX**2)))

# Амплитудный и фазовый спектры
amplitude_spectrum = np.sqrt(REX**2 + IMX**2)
phase_spectrum = np.arctan2(IMX, REX)

# ----------------------------
# ВИЗУАЛИЗАЦИЯ
# ----------------------------
# Используйте figure()функцию для построения графика. Передайте следующие аргументы:
# title: название вашей линейной диаграммы (необязательно)
# x_axis_label: текстовая метка для размещения на оси X диаграммы (необязательно)
# y_axis_label: текстовая метка для размещения на оси Y диаграммы (необязательно)

# создайте новый график с заголовком и метками осей
p = figure(title="Две снусоиды с одинаковой частотой и начальной фазой ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')

# Первая синусоида (y = sin(x))
p.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p.scatter(N_i, sin_k, size=7, marker="square", legend_label="sin_k", line_color="green", fill_alpha=0.3)
p.line(N_i, sin_k, legend_label="sin_k", line_color="green")

X_axis = np.zeros(N)
p.line(N_i, X_axis, line_color="black")

show(p) 


p2 = figure(title="синус и косинус ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')
# Первая синусоида (y = sin(x))
p2.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p2.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p2.scatter(N_i, cos_k, size=7, marker="square", legend_label="cos_k", line_color="green", fill_alpha=0.3)
p2.line(N_i, cos_k, legend_label="cos_k", line_color="green")

X_axis = np.zeros(N)
p2.line(N_i, X_axis, line_color="black")

show(p2)

amp_plot = figure(title="Спект частот ", x_axis_label='Частота', y_axis_label='Амплитуда')
# p.line(N_i_half, amplitude_spectrum, legend_label="sum_wave", line_color="blue")

X_axis = np.zeros(N // 2 + 1)
amp_plot.line(N_i_half, X_axis, line_color="black")

amp_plot.circle(N_i_half, amplitude_spectrum, size=8, color="navy", alpha=0.7)
amp_plot.segment(x0 = N_i_half, y0 = X_axis, x1=N_i_half, y1=amplitude_spectrum, color="navy", alpha=0.6, line_width=2)

show(amp_plot) 


# -----------------------------------------------------------------------------------------------------------------
# График для wave_1 с sin_k
p_wave1 = figure(title="Wave_1 и sin_k", width=1000, height=400,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
p_wave1.scatter(N_i, wave_1, size=8, color="blue", alpha=0.7, legend_label="wave_1")
p_wave1.line(N_i, wave_1, line_color="blue", line_width=2)
p_wave1.scatter(N_i, sin_k, size=8, color="green", alpha=0.7, legend_label="sin_k")
p_wave1.line(N_i, sin_k, line_color="green", line_width=2)
p_wave1.line(N_i, np.zeros(N), line_color="black")
# p_wave1.legend.location = "top_left"
show(p_wave1)
# График для wave_2 с sin_k
p_wave2 = figure(title="Wave_2 и sin_k", width=1000, height=400,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
p_wave2.scatter(N_i, wave_2, size=8, color="red", alpha=0.7, legend_label="wave_2")
p_wave2.line(N_i, wave_2, line_color="red", line_width=2)
p_wave2.scatter(N_i, sin_k, size=8, color="green", alpha=0.7, legend_label="sin_k")
p_wave2.line(N_i, sin_k, line_color="green", line_width=2)
p_wave2.line(N_i, np.zeros(N), line_color="black")
# p_wave2.legend.location = "top_left"

# # Показываем графики в столбце
# show(column(p_wave1, p_wave2))
show(p_wave2)


Loading BokehJS ...

K = 2   I = 0   IMX[K] = 0.00000    sin_mg = 0.00000    REX[K] = 1.00000    cos_mg = 1.00000    sum[I] = 1.00000    np.sin(angle) = 0.00000  np.cos(angle) = 1.00000  angle = 0.00°
K = 2   I = 1   IMX[K] = 0.65328    sin_mg = 0.65328    REX[K] = 1.65328    cos_mg = 0.65328    sum[I] = 0.92388    np.sin(angle) = 0.70711  np.cos(angle) = 0.70711  angle = 45.02°
K = 2   I = 2   IMX[K] = 1.36039    sin_mg = 0.70711    REX[K] = 1.65328    cos_mg = 0.00000    sum[I] = 0.70711    np.sin(angle) = 1.00000  np.cos(angle) = 0.00000  angle = 90.05°
K = 2   I = 3   IMX[K] = 1.63099    sin_mg = 0.27060    REX[K] = 1.38268    cos_mg = -0.27060   sum[I] = 0.38268    np.sin(angle) = 0.70711  np.cos(angle) = -0.70711 angle = 135.07°
K = 2   I = 4   IMX[K] = 1.63099    sin_mg = 0.00000    REX[K] = 1.38268    cos_mg = -0.00000   sum[I] = 0.00000    np.sin(angle) = 0.00000  np.cos(angle) = -1.00000 angle = 180.09°
K = 2   I = 5   IMX[K] = 1.90158    sin_mg = 0.27060    REX[K] = 1.65328    cos_mg = 0.27060  

In [ ]:
!)В формуле IMX[K] -= sum_wave[I] * np.sin(angle) плюс заменяется на минус. причина должна быть понятна при изучении комплексной формы 
преобразования фурье. С минусом вектор синуса смещается на 180° и только после этого векторы синуса и косинуса находятся в пределах 90°.

In [43]:
# Пример постоянной 


import numpy as np
import matplotlib.pyplot as plt
# библиотека на основе которой были сделаны графики
# https://docs.bokeh.org/en/latest/docs/user_guide/basic/layouts.html
from bokeh.layouts import column
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
# Активируем вывод графиков в ноутбук
output_notebook()

# ----------------------------
# ПАРАМЕТРЫ СИГНАЛОВ
# ----------------------------

# Общее количество отсчетов для всех сигналов
N = 32  # Количество дискретных точек (отсчетов) для построения сигналов

# Амплитуды сигналов:
W1_A = 1.0  # Амплитуда дополнительной волны 1
W2_A = 1.0  # Амплитуда дополнительной волны 2
W3_A = 1.0  # Амплитуда дополнительной волны 3
W4_A = 1.0  # Амплитуда дополнительной волны 4

# Фазы сигналов в градусах:
W1_P_DEG = 0   # Фаза дополнительной волны 1
W2_P_DEG = 0   # Фаза дополнительной волны 2
W3_P_DEG = 0   # Фаза дополнительной волны 3
W4_P_DEG = 0   # Фаза дополнительной волны 4

# Количество периодов сигнала (K):
W1_K = 1  # Дополнительная волна 1
W2_K = 2  # Дополнительная волна 2
W3_K = 3  # Дополнительная волна 3
W4_K = 4  # Дополнительная волна 4

# ----------------------------
# ПОДГОТОВКА ДАННЫХ
# ----------------------------

# Создаем массив отсчетов (от 0 до N-1)
N_i = np.arange(N)  # [0, 1, 2, ..., N-1]
N_i_half = np.arange(N // 2 + 1)

# Конвертация фаз из градусов в радианы (требуется для тригонометрических функций)
W1_P_RAD = W1_P_DEG * np.pi / 180 # Формула преобразования: радианы = градусы * π/180
W2_P_RAD = W2_P_DEG * np.pi / 180
W3_P_RAD = W3_P_DEG * np.pi / 180
W4_P_RAD = W4_P_DEG * np.pi / 180

# ----------------------------
# ГЕНЕРАЦИЯ СИГНАЛОВ
# ----------------------------

# Генерация синусоиды:
# Формула: A * sin(2π * K * n/N + φ)
#   K - количество периодов
#   n - текущий отсчет
#   N - общее количество отсчетов
#   φ - фаза в радианах

# Используем sin вместо cos, но с фазой, отличающейся на π/2
# (cos(x) = sin(x + π/2), но здесь фаза задается параметром)
# Генерация  волн:
wave_1 = W1_A * np.sin(2 * np.pi * W1_K * N_i / N + W1_P_RAD)
wave_2 = W2_A * np.sin(2 * np.pi * W2_K * N_i / N + W2_P_RAD)
wave_3 = W3_A * np.sin(2 * np.pi * W3_K * N_i / N + W3_P_RAD)
wave_4 = W4_A * np.sin(2 * np.pi * W4_K * N_i / N + W4_P_RAD)

# ----------------------------
# СУММИРОВАНИЕ СИГНАЛОВ
# ----------------------------

# Сумма сгенерированных сигналов, в зависимости от примера количество сигналов может быть больше или меньше.
# sum_wave = wave_1 + wave_2 + wave_3 + wave_4
sum_wave = wave_1

# Произвести анализ частотного базиса. С выводом данных о нем.
F_b = 2
# DFT (прямое преобразование Фурье)
REX = np.zeros(N // 2 + 1)
IMX = np.zeros(N // 2 + 1)
sin_k = np.zeros(N)
cos_k = np.zeros(N)

for I in range(N):
    sum_wave[I] = 0.5

for K in range(N // 2 + 1):
    for I in range(N):
        angle = 2 * np.pi * K * I / N
        REX[K] += sum_wave[I] * np.cos(angle)
        IMX[K] -= sum_wave[I] * np.sin(angle)
           
        if K == F_b:
            cos_k[I] = np.cos(angle)
            sin_k[I] = np.sin(angle)
            
            # sinus[I] =        np.sin(angle)
            # Форматирование значений для выравнивания
            k_str = f"K = {K}"
            i_str = f"I = {I}"
            imx_str = f"IMX[K] = {IMX[K]:.5f}"
            rex_str = f"REX[K] = {REX[K]:.5f}"
            xx_str = f"sum[I] = {sum_wave[I]:.5f}"
            sin_str = f"np.sin(angle) = {np.sin(angle):.5f}"
            cos_str = f"np.cos(angle) = {np.cos(angle):.5f}"
            angle_str = f"angle = {(180 * angle / 3.14) % 360:.2f}°"
            # angle_str = f"{angle:.2f}"
            
            # Вывод с фиксированной шириной колонок
            print(f"{k_str:<8}{i_str:<8}{imx_str:<20}{rex_str:<20}{xx_str:<20}{sin_str:<25}{cos_str:<25}{angle_str:<5}")
print("Амплитудный и фазовый спектры = " + str(np.sqrt(REX**2 + IMX**2)))

# Амплитудный и фазовый спектры
amplitude_spectrum = np.sqrt(REX**2 + IMX**2)
phase_spectrum = np.arctan2(IMX, REX)

# ----------------------------
# ВИЗУАЛИЗАЦИЯ
# ----------------------------
# Используйте figure()функцию для построения графика. Передайте следующие аргументы:
# title: название вашей линейной диаграммы (необязательно)
# x_axis_label: текстовая метка для размещения на оси X диаграммы (необязательно)
# y_axis_label: текстовая метка для размещения на оси Y диаграммы (необязательно)

# создайте новый график с заголовком и метками осей
p = figure(title="Две снусоиды с одинаковой частотой и начальной фазой ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')

# Первая синусоида (y = sin(x))
p.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p.scatter(N_i, sin_k, size=7, marker="square", legend_label="sin_k", line_color="green", fill_alpha=0.3)
p.line(N_i, sin_k, legend_label="sin_k", line_color="green")

X_axis = np.zeros(N)
p.line(N_i, X_axis, line_color="black")

show(p) 


p2 = figure(title="синус и косинус ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')
# Первая синусоида (y = sin(x))
p2.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p2.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p2.scatter(N_i, cos_k, size=7, marker="square", legend_label="cos_k", line_color="green", fill_alpha=0.3)
p2.line(N_i, cos_k, legend_label="cos_k", line_color="green")

X_axis = np.zeros(N)
p2.line(N_i, X_axis, line_color="black")

show(p2)

amp_plot = figure(title="Спект частот ", x_axis_label='Частота', y_axis_label='Амплитуда')
# p.line(N_i_half, amplitude_spectrum, legend_label="sum_wave", line_color="blue")

X_axis = np.zeros(N // 2 + 1)
amp_plot.line(N_i_half, X_axis, line_color="black")

amp_plot.circle(N_i_half, amplitude_spectrum, size=8, color="navy", alpha=0.7)
amp_plot.segment(x0 = N_i_half, y0 = X_axis, x1=N_i_half, y1=amplitude_spectrum, color="navy", alpha=0.6, line_width=2)

show(amp_plot) 

Loading BokehJS ...

K = 2   I = 0   IMX[K] = 0.00000    REX[K] = 0.50000    sum[I] = 0.50000    np.sin(angle) = 0.00000  np.cos(angle) = 1.00000  angle = 0.00°
K = 2   I = 1   IMX[K] = -0.19134   REX[K] = 0.96194    sum[I] = 0.50000    np.sin(angle) = 0.38268  np.cos(angle) = 0.92388  angle = 22.51°
K = 2   I = 2   IMX[K] = -0.54490   REX[K] = 1.31549    sum[I] = 0.50000    np.sin(angle) = 0.70711  np.cos(angle) = 0.70711  angle = 45.02°
K = 2   I = 3   IMX[K] = -1.00683   REX[K] = 1.50683    sum[I] = 0.50000    np.sin(angle) = 0.92388  np.cos(angle) = 0.38268  angle = 67.53°
K = 2   I = 4   IMX[K] = -1.50683   REX[K] = 1.50683    sum[I] = 0.50000    np.sin(angle) = 1.00000  np.cos(angle) = 0.00000  angle = 90.05°
K = 2   I = 5   IMX[K] = -1.96877   REX[K] = 1.31549    sum[I] = 0.50000    np.sin(angle) = 0.92388  np.cos(angle) = -0.38268 angle = 112.56°
K = 2   I = 6   IMX[K] = -2.32233   REX[K] = 0.96194    sum[I] = 0.50000    np.sin(angle) = 0.70711  np.cos(angle) = -0.70711 angle = 135.07°
K = 2   I = 

In [ ]:
!) Нельзя задавать частот сигнала ниже минимальной базовой частоты F_b. она не помещается на графике.Будет производится анализ части сигнала.
    
!) Если частота сигнала попадает между базовыми частотами то корреляцию достоверно не получится вычислить. Например базовая частота равна 1 а частота
   сигнал 1.5 то на спекте пик будет 1Гц а 0Гц и 2Гц поменьше и последующие частоты будут умеьшатся 
!)Если сигнал постоянный то все частоты кроме нулевой будут равны нулю. начнем с синусоиды на графе видно что от 0° до 180° 
  синус положителена а от 180° до 360° отрицательн. и первый и второй учаток синусоиды имеют одинаковую площадь. но у первого участка все
  отсчеты положильны а у второго все отрицательные. после корреляции участки  REX и
  IMX взаимно вычтутся.

  Нулевая частота не равна нулю потомучто cos(0°) = 1. по этой причине отсчеты сигнала просто будут складываться

In [17]:

import numpy as np
import matplotlib.pyplot as plt
# библиотека на основе которой были сделаны графики
# https://docs.bokeh.org/en/latest/docs/user_guide/basic/layouts.html
from bokeh.layouts import column
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
# Активируем вывод графиков в ноутбук
output_notebook()

# ----------------------------
# ПАРАМЕТРЫ СИГНАЛОВ
# ----------------------------

# Общее количество отсчетов для всех сигналов
N = 16  # Количество дискретных точек (отсчетов) для построения сигналов

# Амплитуды сигналов:
W1_A = 1.0  # Амплитуда дополнительной волны 1
W2_A = 1.0  # Амплитуда дополнительной волны 2
W3_A = 1.0  # Амплитуда дополнительной волны 3
W4_A = 1.0  # Амплитуда дополнительной волны 4

# Фазы сигналов в градусах:
W1_P_DEG = 0   # Фаза дополнительной волны 1
W2_P_DEG = 0   # Фаза дополнительной волны 2
W3_P_DEG = 0   # Фаза дополнительной волны 3
W4_P_DEG = 0   # Фаза дополнительной волны 4

# Количество периодов сигнала (K):
W1_K = 0.1  # Дополнительная волна 1
W2_K = 2  # Дополнительная волна 2
W3_K = 3  # Дополнительная волна 3
W4_K = 4  # Дополнительная волна 4

# ----------------------------
# ПОДГОТОВКА ДАННЫХ
# ----------------------------

# Создаем массив отсчетов (от 0 до N-1)
N_i = np.arange(N)  # [0, 1, 2, ..., N-1]
N_i_half = np.arange(N // 2 + 1)

# Конвертация фаз из градусов в радианы (требуется для тригонометрических функций)
W1_P_RAD = W1_P_DEG * np.pi / 180 # Формула преобразования: радианы = градусы * π/180
W2_P_RAD = W2_P_DEG * np.pi / 180
W3_P_RAD = W3_P_DEG * np.pi / 180
W4_P_RAD = W4_P_DEG * np.pi / 180

# ----------------------------
# ГЕНЕРАЦИЯ СИГНАЛОВ
# ----------------------------

# Генерация синусоиды:
# Формула: A * sin(2π * K * n/N + φ)
#   K - количество периодов
#   n - текущий отсчет
#   N - общее количество отсчетов
#   φ - фаза в радианах

# Используем sin вместо cos, но с фазой, отличающейся на π/2
# (cos(x) = sin(x + π/2), но здесь фаза задается параметром)
# Генерация  волн:
wave_1 = W1_A * np.sin(2 * np.pi * W1_K * N_i / N + W1_P_RAD)
wave_2 = W2_A * np.sin(2 * np.pi * W2_K * N_i / N + W2_P_RAD)
wave_3 = W3_A * np.sin(2 * np.pi * W3_K * N_i / N + W3_P_RAD)
wave_4 = W4_A * np.sin(2 * np.pi * W4_K * N_i / N + W4_P_RAD)

# ----------------------------
# СУММИРОВАНИЕ СИГНАЛОВ
# ----------------------------

# Сумма сгенерированных сигналов, в зависимости от примера количество сигналов может быть больше или меньше.
# sum_wave = wave_1 + wave_2 + wave_3 + wave_4
sum_wave = wave_1

# Произвести анализ частотного базиса. С выводом данных о нем.
F_b = 2
# DFT (прямое преобразование Фурье)
REX = np.zeros(N // 2 + 1)
IMX = np.zeros(N // 2 + 1)
sin_k = np.zeros(N)
cos_k = np.zeros(N)

for K in range(N // 2 + 1):
    for I in range(N):
        angle = 2 * np.pi * K * I / N
        REX[K] += sum_wave[I] * np.cos(angle)
        IMX[K] -= sum_wave[I] * np.sin(angle)
           
        if K == F_b:
            cos_k[I] = np.cos(angle)
            sin_k[I] = np.sin(angle)
            
            # sinus[I] =        np.sin(angle)
            # Форматирование значений для выравнивания
            k_str = f"K = {K}"
            i_str = f"I = {I}"
            imx_str = f"IMX[K] = {IMX[K]:.5f}"
            rex_str = f"REX[K] = {REX[K]:.5f}"
            xx_str = f"sum[I] = {sum_wave[I]:.5f}"
            sin_str = f"np.sin(angle) = {np.sin(angle):.5f}"
            cos_str = f"np.cos(angle) = {np.cos(angle):.5f}"
            angle_str = f"angle = {(180 * angle / 3.14) % 360:.2f}°"
            # angle_str = f"{angle:.2f}"
            
            # Вывод с фиксированной шириной колонок
            print(f"{k_str:<8}{i_str:<8}{imx_str:<20}{rex_str:<20}{xx_str:<20}{sin_str:<25}{cos_str:<25}{angle_str:<5}")
print("Амплитудный и фазовый спектры = " + str(np.sqrt(REX**2 + IMX**2)))

# Амплитудный и фазовый спектры
amplitude_spectrum = np.sqrt(REX**2 + IMX**2)
phase_spectrum = np.arctan2(IMX, REX)

# ----------------------------
# ВИЗУАЛИЗАЦИЯ
# ----------------------------
# Используйте figure()функцию для построения графика. Передайте следующие аргументы:
# title: название вашей линейной диаграммы (необязательно)
# x_axis_label: текстовая метка для размещения на оси X диаграммы (необязательно)
# y_axis_label: текстовая метка для размещения на оси Y диаграммы (необязательно)

# создайте новый график с заголовком и метками осей
p = figure(title="Две снусоиды с одинаковой частотой и начальной фазой ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')

# Первая синусоида (y = sin(x))
p.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p.scatter(N_i, sin_k, size=7, marker="square", legend_label="sin_k", line_color="green", fill_alpha=0.3)
p.line(N_i, sin_k, legend_label="sin_k", line_color="green")

X_axis = np.zeros(N)
p.line(N_i, X_axis, line_color="black")

show(p) 


p2 = figure(title="синус и косинус ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')
# Первая синусоида (y = sin(x))
p2.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p2.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p2.scatter(N_i, cos_k, size=7, marker="square", legend_label="cos_k", line_color="green", fill_alpha=0.3)
p2.line(N_i, cos_k, legend_label="cos_k", line_color="green")

X_axis = np.zeros(N)
p2.line(N_i, X_axis, line_color="black")

show(p2)

amp_plot = figure(title="Спект частот ", x_axis_label='Частота', y_axis_label='Амплитуда')
# p.line(N_i_half, amplitude_spectrum, legend_label="sum_wave", line_color="blue")

X_axis = np.zeros(N // 2 + 1)
amp_plot.line(N_i_half, X_axis, line_color="black")

amp_plot.circle(N_i_half, amplitude_spectrum, size=8, color="navy", alpha=0.7)
amp_plot.segment(x0 = N_i_half, y0 = X_axis, x1=N_i_half, y1=amplitude_spectrum, color="navy", alpha=0.6, line_width=2)

show(amp_plot) 

Loading BokehJS ...

K = 2   I = 0   IMX[K] = 0.00000    REX[K] = 0.00000    sum[I] = 0.00000    np.sin(angle) = 0.00000  np.cos(angle) = 1.00000  angle = 0.00°
K = 2   I = 1   IMX[K] = -0.02776   REX[K] = 0.02776    sum[I] = 0.03926    np.sin(angle) = 0.70711  np.cos(angle) = 0.70711  angle = 45.02°
K = 2   I = 2   IMX[K] = -0.10622   REX[K] = 0.02776    sum[I] = 0.07846    np.sin(angle) = 1.00000  np.cos(angle) = 0.00000  angle = 90.05°
K = 2   I = 3   IMX[K] = -0.18933   REX[K] = -0.05535   sum[I] = 0.11754    np.sin(angle) = 0.70711  np.cos(angle) = -0.70711 angle = 135.07°
K = 2   I = 4   IMX[K] = -0.18933   REX[K] = -0.21179   sum[I] = 0.15643    np.sin(angle) = 0.00000  np.cos(angle) = -1.00000 angle = 180.09°
K = 2   I = 5   IMX[K] = -0.05138   REX[K] = -0.34973   sum[I] = 0.19509    np.sin(angle) = -0.70711 np.cos(angle) = -0.70711 angle = 225.11°
K = 2   I = 6   IMX[K] = 0.18206    REX[K] = -0.34973   sum[I] = 0.23345    np.sin(angle) = -1.00000 np.cos(angle) = -0.00000 angle = 270.14°
K = 2   I 

In [14]:

import numpy as np
import matplotlib.pyplot as plt
# библиотека на основе которой были сделаны графики
# https://docs.bokeh.org/en/latest/docs/user_guide/basic/layouts.html
from bokeh.layouts import column
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
# Активируем вывод графиков в ноутбук
output_notebook()

# ----------------------------
# ПАРАМЕТРЫ СИГНАЛОВ
# ----------------------------

# Общее количество отсчетов для всех сигналов
N = 51  # Количество дискретных точек (отсчетов) для построения сигналов

# Амплитуды сигналов:
W1_A = 1.0  # Амплитуда дополнительной волны 1
W2_A = 2.0  # Амплитуда дополнительной волны 2
W3_A = 3.0  # Амплитуда дополнительной волны 3
W4_A = 4.0  # Амплитуда дополнительной волны 4

# Фазы сигналов в градусах:
W1_P_DEG = 0   # Фаза дополнительной волны 1
W2_P_DEG = 0   # Фаза дополнительной волны 2
W3_P_DEG = 0   # Фаза дополнительной волны 3
W4_P_DEG = 0   # Фаза дополнительной волны 4

# Количество периодов сигнала (K):
W1_K = 25  # Дополнительная волна 1
W2_K = 0  # Дополнительная волна 2
W3_K = 0  # Дополнительная волна 3
W4_K = 0 # Дополнительная волна 4

# ----------------------------
# ПОДГОТОВКА ДАННЫХ
# ----------------------------

# Создаем массив отсчетов (от 0 до N-1)
N_i = np.arange(N)  # [0, 1, 2, ..., N-1]
N_i_half = np.arange(N // 2 + 1)

# Конвертация фаз из градусов в радианы (требуется для тригонометрических функций)
W1_P_RAD = W1_P_DEG * np.pi / 180 # Формула преобразования: радианы = градусы * π/180
W2_P_RAD = W2_P_DEG * np.pi / 180
W3_P_RAD = W3_P_DEG * np.pi / 180
W4_P_RAD = W4_P_DEG * np.pi / 180

# ----------------------------
# ГЕНЕРАЦИЯ СИГНАЛОВ
# ----------------------------

# Генерация синусоиды:
# Формула: A * sin(2π * K * n/N + φ)
#   K - количество периодов
#   n - текущий отсчет
#   N - общее количество отсчетов
#   φ - фаза в радианах

# Используем sin вместо cos, но с фазой, отличающейся на π/2
# (cos(x) = sin(x + π/2), но здесь фаза задается параметром)
# Генерация  волн:
wave_1 = W1_A * np.sin(2 * np.pi * W1_K * N_i / N + W1_P_RAD)
wave_2 = W2_A * np.sin(2 * np.pi * W2_K * N_i / N + W2_P_RAD)
wave_3 = W3_A * np.sin(2 * np.pi * W3_K * N_i / N + W3_P_RAD)
wave_4 = W4_A * np.sin(2 * np.pi * W4_K * N_i / N + W4_P_RAD)

# ----------------------------
# СУММИРОВАНИЕ СИГНАЛОВ
# ----------------------------

# Сумма сгенерированных сигналов, в зависимости от примера количество сигналов может быть больше или меньше.
sum_wave = wave_1 + wave_2 + wave_3 + wave_4


# Произвести анализ частотного базиса. С выводом данных о нем.
F_b = 1
# DFT (прямое преобразование Фурье)
REX = np.zeros(N // 2 + 1)
IMX = np.zeros(N // 2 + 1)
sin_k = np.zeros(N)
cos_k = np.zeros(N)

for K in range(N // 2 + 1):
    for I in range(N):
        angle = 2 * np.pi * K * I / N
        REX[K] += sum_wave[I] * np.cos(angle)
        IMX[K] -= sum_wave[I] * np.sin(angle)
        # IMX[K] += sum_wave[I] * np.sin(angle)

        if K == F_b:
            cos_k[I] = np.cos(angle)
            sin_k[I] = np.sin(angle)
            
            # sinus[I] =        np.sin(angle)
            # Форматирование значений для выравнивания
            k_str = f"K = {K}"
            i_str = f"I = {I}"
            imx_str = f"IMX[K] = {IMX[K]:.5f}"
            rex_str = f"REX[K] = {REX[K]:.5f}"
            sin_mg = f"sin_mg = {sum_wave[I] * np.sin(angle):.5f}"
            cos_mg = f"cos_mg = {sum_wave[I] * np.cos(angle):.5f}"
            xx_str = f"sum[I] = {sum_wave[I]:.5f}"
            sin_str = f"np.sin(angle) = {np.sin(angle):.5f}"
            cos_str = f"np.cos(angle) = {np.cos(angle):.5f}"
            angle_str = f"angle = {(180 * angle / 3.14) % 360:.2f}°"
            # angle_str = f"{angle:.2f}"
 
            # Вывод с фиксированной шириной колонок
            print(f"{k_str:<8}{i_str:<8}{imx_str:<20}{sin_mg:<20}{rex_str:<20}{cos_mg:<20}{xx_str:<20}{sin_str:<25}{cos_str:<25}{angle_str:<5}")
print("Амплитудный и фазовый спектры = " + str(np.sqrt(REX**2 + IMX**2)))

# Амплитудный и фазовый спектры
amplitude_spectrum = np.sqrt(REX**2 + IMX**2)
phase_spectrum = np.arctan2(IMX, REX)

# ----------------------------
# ВИЗУАЛИЗАЦИЯ
# ----------------------------
# Используйте figure()функцию для построения графика. Передайте следующие аргументы:
# title: название вашей линейной диаграммы (необязательно)
# x_axis_label: текстовая метка для размещения на оси X диаграммы (необязательно)
# y_axis_label: текстовая метка для размещения на оси Y диаграммы (необязательно)

# создайте новый график с заголовком и метками осей
p = figure(title="Две снусоиды с одинаковой частотой и начальной фазой ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')

# Первая синусоида (y = sin(x))
p.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p.scatter(N_i, sin_k, size=7, marker="square", legend_label="sin_k", line_color="green", fill_alpha=0.3)
p.line(N_i, sin_k, legend_label="sin_k", line_color="green")

X_axis = np.zeros(N)
p.line(N_i, X_axis, line_color="black")

show(p) 


p2 = figure(title="синус и косинус ", x_axis_label='Отсчеты', y_axis_label='Амплитуда')
# Первая синусоида (y = sin(x))
p2.scatter(N_i, sum_wave, size=7, marker="circle", legend_label="sum_wave", line_color="blue", fill_alpha=0.3)
p2.line(N_i, sum_wave, legend_label="sum_wave", line_color="blue")

p2.scatter(N_i, cos_k, size=7, marker="square", legend_label="cos_k", line_color="green", fill_alpha=0.3)
p2.line(N_i, cos_k, legend_label="cos_k", line_color="green")

X_axis = np.zeros(N)
p2.line(N_i, X_axis, line_color="black")

show(p2)

amp_plot = figure(title="Спект частот ", x_axis_label='Частота', y_axis_label='Амплитуда')
# p.line(N_i_half, amplitude_spectrum, legend_label="sum_wave", line_color="blue")

X_axis = np.zeros(N // 2 + 1)
amp_plot.line(N_i_half, X_axis, line_color="black")

amp_plot.circle(N_i_half, amplitude_spectrum, size=8, color="navy", alpha=0.7)
amp_plot.segment(x0 = N_i_half, y0 = X_axis, x1=N_i_half, y1=amplitude_spectrum, color="navy", alpha=0.6, line_width=2)

show(amp_plot) 


# -----------------------------------------------------------------------------------------------------------------
# График для wave_1 с sin_k
p_wave1 = figure(title="Wave_1 и sin_k", width=1000, height=400,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
p_wave1.scatter(N_i, wave_1, size=8, color="blue", alpha=0.7, legend_label="wave_1")
p_wave1.line(N_i, wave_1, line_color="blue", line_width=2)
p_wave1.scatter(N_i, sin_k, size=8, color="green", alpha=0.7, legend_label="sin_k")
p_wave1.line(N_i, sin_k, line_color="green", line_width=2)
p_wave1.line(N_i, np.zeros(N), line_color="black")
# p_wave1.legend.location = "top_left"
show(p_wave1)
# График для wave_2 с sin_k
p_wave2 = figure(title="Wave_2 и sin_k", width=1000, height=400,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
p_wave2.scatter(N_i, wave_2, size=8, color="red", alpha=0.7, legend_label="wave_2")
p_wave2.line(N_i, wave_2, line_color="red", line_width=2)
p_wave2.scatter(N_i, sin_k, size=8, color="green", alpha=0.7, legend_label="sin_k")
p_wave2.line(N_i, sin_k, line_color="green", line_width=2)
p_wave2.line(N_i, np.zeros(N), line_color="black")
# p_wave2.legend.location = "top_left"

# # Показываем графики в столбце
# show(column(p_wave1, p_wave2))
show(p_wave2)

Loading BokehJS ...

K = 1   I = 0   IMX[K] = 0.00000    sin_mg = 0.00000    REX[K] = 0.00000    cos_mg = 0.00000    sum[I] = 0.00000    np.sin(angle) = 0.00000  np.cos(angle) = 1.00000  angle = 0.00°
K = 1   I = 1   IMX[K] = -0.00757   sin_mg = 0.00757    REX[K] = 0.06109    cos_mg = 0.06109    sum[I] = 0.06156    np.sin(angle) = 0.12289  np.cos(angle) = 0.99242  angle = 7.06°
K = 1   I = 2   IMX[K] = 0.02241    sin_mg = -0.02997   REX[K] = -0.05808   cos_mg = -0.11918   sum[I] = -0.12289   np.sin(angle) = 0.24391  np.cos(angle) = 0.96980  angle = 14.12°
K = 1   I = 3   IMX[K] = -0.04397   sin_mg = 0.06638    REX[K] = 0.11326    cos_mg = 0.17134    sum[I] = 0.18375    np.sin(angle) = 0.36124  np.cos(angle) = 0.93247  angle = 21.19°
K = 1   I = 4   IMX[K] = 0.07143    sin_mg = -0.11539   REX[K] = -0.10163   cos_mg = -0.21489   sum[I] = -0.24391   np.sin(angle) = 0.47309  np.cos(angle) = 0.88101  angle = 28.25°
K = 1   I = 5   IMX[K] = -0.10373   sin_mg = 0.17515    REX[K] = 0.14580    cos_mg = 0.24743    s